# Các tiêu chí chia nhánh: Information Gain, Gain Ratio, Gini

*Bài 6, mục 6.1.3.* ID3 dùng Information Gain, C4.5 dùng Gain Ratio, CART dùng chỉ số Gini và luôn chia nhị phân.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path('..') / 'data'   # notebook chạy trong codes/
df = pd.read_csv(DATA / 'weather.csv')


def entropy(labels):
    """Entropy H(S) của một dãy nhãn (log cơ số 2)."""
    p = pd.Series(labels).value_counts(normalize=True).values   # tỉ lệ mỗi lớp p_i
    return 0.0 - np.sum(p * np.log2(p))     # 0.0 - ... để tránh in ra -0.000


def information_gain(data, attribute, target='play'):
    """IG(S, A) = H(S) - tổng có trọng số entropy của các tập con S_v."""
    weighted = sum(len(s) / len(data) * entropy(s[target]) for _, s in data.groupby(attribute))
    return entropy(data[target]) - weighted


def split_info(data, attribute):
    """SplitInfo(S, A) = -tổng |S_v|/|S| log2(|S_v|/|S|)."""
    return entropy(data[attribute])


def gain_ratio(data, attribute, target='play'):
    return information_gain(data, attribute, target) / split_info(data, attribute)


def gini(labels):
    """Chỉ số Gini(S) = 1 - tổng p_i^2."""
    p = pd.Series(labels).value_counts(normalize=True).values
    return 1 - np.sum(p**2)

## 1. Nhược điểm của Information Gain: ưu tiên thuộc tính nhiều giá trị

Cột `day` (số thứ tự ngày) có 10 giá trị khác nhau, mỗi tập con chỉ có 1 mẫu nên tinh khiết: IG đạt cực đại bằng $H(S) = 1$, nhưng thuộc tính này vô nghĩa khi dự đoán ngày mới. Gain Ratio chia IG cho SplitInfo (entropy của chính thuộc tính) để phạt các thuộc tính chia quá vụn.

In [2]:
rows = []
for a in ['day', 'weather', 'temperature', 'humidity', 'wind']:
    rows.append([a, df[a].nunique(), information_gain(df, a), split_info(df, a), gain_ratio(df, a)])
table = pd.DataFrame(rows, columns=['thuộc tính', 'số giá trị', 'IG', 'SplitInfo', 'GainRatio'])
table.round(3)

,thuộc tính,số giá trị,IG,SplitInfo,GainRatio
0,day,10,1.000,3.322,0.301
1,weather,3,0.400,1.571,0.255
2,temperature,3,0.115,1.361,0.084
3,humidity,2,0.035,0.881,0.040
4,wind,2,0.125,0.971,0.128


IG của `day` gấp 2.5 lần IG của `weather`, nhưng Gain Ratio của `day` chỉ còn gấp khoảng 1.2 lần: SplitInfo đã phạt mạnh thuộc tính chia vụn. Với chỉ 10 mẫu, mức phạt chưa đủ để đảo ngược thứ tự, nên trong thực tế vẫn phải loại các cột định danh (mã, số thứ tự) ngay từ bước tiền xử lý.

## 2. Chỉ số Gini

In [3]:
print('Gini(6A, 4B)  = %.2f' % gini(['A'] * 6 + ['B'] * 4))
print('Gini(5A, 5B)  = %.2f' % gini(['A'] * 5 + ['B'] * 5))
print('Gini(10A, 0B) = %.2f' % gini(['A'] * 10))
print('Gini(S) của dữ liệu thời tiết = %.2f' % gini(df['play']))

Gini(6A, 4B)  = 0.48
Gini(5A, 5B)  = 0.50
Gini(10A, 0B) = 0.00
Gini(S) của dữ liệu thời tiết = 0.50


## 3. CART: phép chia nhị phân tốt nhất

Với thuộc tính rời rạc, CART xét các cách chia tập giá trị thành hai nhóm. Ở đây xét các phép chia dạng $\{v\}$ | phần còn lại (với thuộc tính 3 giá trị, đây là cả 3 cách chia), chọn phép chia có Gini có trọng số nhỏ nhất, tức độ giảm Gini lớn nhất.

In [4]:
def weighted_gini(data, mask, target='play'):
    left, right = data[mask], data[~mask]
    return len(left) / len(data) * gini(left[target]) + len(right) / len(data) * gini(right[target])


rows = []
for a in ['weather', 'temperature', 'humidity', 'wind']:
    values = sorted(df[a].unique())
    if len(values) == 2:
        values = values[:1]   # thuộc tính 2 giá trị: chỉ có một cách chia
    for value in values:
        g = weighted_gini(df, df[a] == value)
        rows.append([a, '{%s} | còn lại' % value, g, gini(df['play']) - g])
splits = pd.DataFrame(rows, columns=['thuộc tính', 'phép chia', 'Gini sau chia', 'độ giảm Gini'])
splits.sort_values('độ giảm Gini', ascending=False).round(3)

,thuộc tính,phép chia,Gini sau chia,độ giảm Gini
0,weather,{Cloudy} | còn lại,0.286,0.214
1,weather,{Rainy} | còn lại,0.417,0.083
7,wind,{Strong} | còn lại,0.417,0.083
3,temperature,{Cool} | còn lại,0.444,0.056
2,weather,{Sunny} | còn lại,0.476,0.024
6,humidity,{High} | còn lại,0.476,0.024
5,temperature,{Mild} | còn lại,0.480,0.020
4,temperature,{Hot} | còn lại,0.500,0.000


## 4. Thuộc tính liên tục: tìm ngưỡng chia

C4.5 và CART xử lý thuộc tính số bằng phép kiểm tra $x \le t$. Các ngưỡng ứng viên là trung điểm giữa hai giá trị liên tiếp (sau khi sắp xếp). Ví dụ: giả sử đo được nhiệt độ (°C) của 10 ngày.

In [5]:
temperature_c = np.array([30, 31, 24, 22, 21, 15, 23, 32, 29, 20])    # nhiệt độ °C
play = df['play'].values
values = np.sort(np.unique(temperature_c))
thresholds = (values[:-1] + values[1:]) / 2      # trung điểm hai giá trị liên tiếp
H = entropy(play)
best = None
for t in thresholds:
    left, right = play[temperature_c <= t], play[temperature_c > t]
    ig = H - len(left) / len(play) * entropy(left) - len(right) / len(play) * entropy(right)
    ig = max(ig, 0.0)        # tránh giá trị âm rất nhỏ do sai số làm tròn
    if best is None or ig > best[1]:
        best = (t, ig)
    print('t = %4.1f: trái %d mẫu, phải %d mẫu, IG = %.3f' % (t, len(left), len(right), ig))
print('Ngưỡng tốt nhất: nhiệt độ <= %.1f, IG = %.3f' % best)

t = 17.5: trái 1 mẫu, phải 9 mẫu, IG = 0.108
t = 20.5: trái 2 mẫu, phải 8 mẫu, IG = 0.236
t = 21.5: trái 3 mẫu, phải 7 mẫu, IG = 0.396
t = 22.5: trái 4 mẫu, phải 6 mẫu, IG = 0.125
t = 23.5: trái 5 mẫu, phải 5 mẫu, IG = 0.029
t = 26.5: trái 6 mẫu, phải 4 mẫu, IG = 0.000
t = 29.5: trái 7 mẫu, phải 3 mẫu, IG = 0.035
t = 30.5: trái 8 mẫu, phải 2 mẫu, IG = 0.000
t = 31.5: trái 9 mẫu, phải 1 mẫu, IG = 0.108
Ngưỡng tốt nhất: nhiệt độ <= 21.5, IG = 0.396


## 5. scikit-learn: tiêu chí `gini` và `entropy`

`DecisionTreeClassifier` (CART) cho phép chọn `criterion='gini'` (mặc định) hoặc `'entropy'`. Trên dữ liệu nhỏ này hai tiêu chí cho cùng một cây; trong thực tế hai tiêu chí cũng thường cho cây rất giống nhau.

In [6]:
from sklearn.tree import DecisionTreeClassifier, export_text

X = pd.get_dummies(df[['weather', 'temperature', 'humidity', 'wind']], dtype=int)
for criterion in ['gini', 'entropy']:
    tree = DecisionTreeClassifier(criterion=criterion, random_state=0).fit(X, df['play'])
    print('criterion = %s: độ sâu %d, %d nút lá' % (criterion, tree.get_depth(), tree.get_n_leaves()))
    print(export_text(tree, feature_names=list(X.columns)))

criterion = gini: độ sâu 4, 5 nút lá
|--- weather_Cloudy <= 0.50
|   |--- temperature_Mild <= 0.50
|   |   |--- class: No
|   |--- temperature_Mild >  0.50
|   |   |--- humidity_High <= 0.50
|   |   |   |--- class: Yes
|   |   |--- humidity_High >  0.50
|   |   |   |--- wind_Strong <= 0.50
|   |   |   |   |--- class: Yes
|   |   |   |--- wind_Strong >  0.50
|   |   |   |   |--- class: No
|--- weather_Cloudy >  0.50
|   |--- class: Yes

criterion = entropy: độ sâu 4, 5 nút lá
|--- weather_Cloudy <= 0.50
|   |--- temperature_Mild <= 0.50
|   |   |--- class: No
|   |--- temperature_Mild >  0.50
|   |   |--- humidity_High <= 0.50
|   |   |   |--- class: Yes
|   |   |--- humidity_High >  0.50
|   |   |   |--- wind_Strong <= 0.50
|   |   |   |   |--- class: Yes
|   |   |   |--- wind_Strong >  0.50
|   |   |   |   |--- class: No
|--- weather_Cloudy >  0.50
|   |--- class: Yes

